# Урок 9 — Декоратори

> Цей урок будується на Уроці 7 (Функції): ви вже вмієте писати функції з параметрами і `return`. Тут ви навчитесь передавати функції як звичайні значення і "загортати" одну функцію в іншу, щоб додати їй поведінку, не змінюючи її тіло. Повний розгляд функцій як об'єктів першого класу (зберігання в структурах даних, `map`/`filter`, функції вищого порядку як самостійна тема) буде в Модулі 2 — тут лише той мінімум, який потрібен для розуміння декораторів.

Структура уроку: **RETRIEVE → CONCEPT → PREDICT/RUN/INVESTIGATE → MODIFY → CREATE → TRANSFER** (та сама послідовність, що й в Уроках 4 і 7).

## 🔁 RETRIEVE — пригадай Урок 7 (без підглядання)

Дай відповідь усно чи на папері, **не запускаючи нічого**:

1. Чим відрізняється **параметр** від **аргумента**?
2. Функція визначена без жодного `return`. Що опиниться у змінній, якщо зберегти результат її виклику?
3. Навіщо розкладати один суцільний блок коду на кілька функцій, а не тримати все в одному місці?

<details>
<summary>Відповіді</summary>

1. Параметр — ім'я в дужках при *визначенні* функції (`def f(x):`). Аргумент — реальне значення, передане при *виклику* (`f(5)`).
2. `None` — Python автоматично повертає `None`, якщо в тілі функції немає явного `return`.
3. Принцип єдиної відповідальності: кожна функція відповідає за одну задачу, її легше називати, тестувати окремо і повторно використовувати; логіку, яку інакше довелося б дублювати, можна тримати в одному місці.

</details>

## 📖 CONCEPT

### 1. Проблема: перевірка прав доступу, розкидана по коду

Уявіть простий блог із діями над постами: перегляд, створення, редагування, публікація, видалення, архівування. У блозі є три ролі:

- `guest` — гість, може тільки читати;
- `user` — звичайний користувач;
- `admin` — адміністратор, може все.

Кожна дія повинна перевіряти, чи має поточний користувач право її виконати. Найпростіший спосіб — додати `if` на початок кожної функції.

In [1]:
current_user = {}


def view_post(post_id):
    if current_user["role"] not in ["guest", "user", "admin"]:
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} переглядає пост #{post_id}")


def create_post(title):
    if current_user["role"] not in ["user", "admin"]:
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} створює пост: '{title}'")


def edit_post(post_id):
    if current_user["role"] not in ["user", "admin"]:
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} редагує пост #{post_id}")


def publish_post(post_id):
    if current_user["role"] not in ["admin"]:
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} публікує пост #{post_id}")


def delete_post(post_id):
    if current_user["role"] not in ["admin"]:
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} видаляє пост #{post_id}")


def archive_post(post_id):
    if current_user["role"] not in ["admin"]:
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} архівує пост #{post_id}")


current_user = {"name": "Іван", "role": "guest"}
view_post(1)
create_post("Мій перший пост")
delete_post(1)

current_user = {"name": "Оля", "role": "admin"}
view_post(1)
create_post("Важливе оголошення")
delete_post(1)

Іван переглядає пост #1
Доступ заборонено
Доступ заборонено
Оля переглядає пост #1
Оля створює пост: 'Важливе оголошення'
Оля видаляє пост #1


### 2. Ціна дублювання: зміна вимог

Код вище працює, але той самий фрагмент

```python
if current_user["role"] not in [...]:
    print("Доступ заборонено")
    return
```

повторюється в кожній із шести функцій. Це дублювання коду: логіка доступу не зберігається в одному місці, а розкидана по всій програмі.

Наскільки це дорого коштує, видно, щойно змінюються вимоги. Припустимо, потрібно додати нову роль `moderator`, яка може редагувати й публікувати пости, але не може їх видаляти:

| Функція | Хто мав доступ | Хто матиме доступ |
|---------|-----------------|---------------------|
| `view_post` | guest, user, admin | guest, user, admin, **moderator** |
| `create_post` | user, admin | user, admin |
| `edit_post` | user, admin | user, admin, **moderator** |
| `publish_post` | admin | admin, **moderator** |
| `delete_post` | admin | admin |
| `archive_post` | admin | admin |

Щоб унести цю зміну, доведеться вручну відкрити чотири функції і поправити список ролей у кожній із них окремо — і так щоразу, коли бізнес-правила зміняться.

In [2]:
# Ілюстрація ціни зміни: та сама зміна ("додати moderator"),
# внесена вручну в кожну функцію окремо

def view_post_v2(post_id):
    if current_user["role"] not in ["guest", "user", "admin", "moderator"]:  # змінено
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} переглядає пост #{post_id}")


def edit_post_v2(post_id):
    if current_user["role"] not in ["user", "admin", "moderator"]:  # змінено
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} редагує пост #{post_id}")


def publish_post_v2(post_id):
    if current_user["role"] not in ["admin", "moderator"]:  # змінено
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} публікує пост #{post_id}")


print("Чотири функції довелося відредагувати вручну заради однієї нової ролі.")

Чотири функції довелося відредагувати вручну заради однієї нової ролі.


### 3. Місток: функції — це значення

У Python функція — такий самий об'єкт, як число чи рядок. Її можна зберегти у змінній, передати як аргумент іншій функції або повернути з функції як результат.

In [3]:
def say_hello():
    print("Привіт!")


def run_function(func):
    print("--- Запускаємо функцію ---")
    func()
    print("--- Готово ---")


run_function(say_hello)  # передаємо say_hello без дужок — саму функцію, а не результат її виклику

--- Запускаємо функцію ---
Привіт!
--- Готово ---


Функція також може **повертати** іншу функцію — а не тільки приймати її як аргумент.

In [4]:
def create_greeter(name):
    def greet():
        print(f"Привіт, {name}!")
    return greet  # повертаємо функцію greet, не викликаючи її


greet_ivan = create_greeter("Іван")
greet_olia = create_greeter("Оля")

greet_ivan()
greet_olia()

Привіт, Іван!
Привіт, Оля!


### 4. Замикання (closures)

`greet` у прикладі вище звертається до `name` — змінної з оточуючої функції `create_greeter`. Коли `create_greeter` завершує роботу, `name` не зникає: `greet` "пам'ятає" її значення. Функція, яка утримує доступ до змінних з оточуючого середовища навіть після завершення зовнішньої функції, називається **замиканням (closure)**.

Якщо всередині замикання потрібно не просто читати, а **змінювати** змінну з оточуючої функції, звичайного доступу недостатньо — потрібне ключове слово `nonlocal`. Без нього присвоєння всередині вкладеної функції створює нову локальну змінну замість зміни зовнішньої.

In [5]:
def make_counter():
    count = 0

    def increment():
        nonlocal count  # без цього рядка count += 1 впаде з UnboundLocalError
        count += 1
        return count

    return increment


counter_a = make_counter()
counter_b = make_counter()

print(counter_a())  # 1
print(counter_a())  # 2
print(counter_a())  # 3
print(counter_b())  # 1 — counter_b має власне, незалежне замикання над своєю count

1
2
3
1


### 5. Побудова обгортки вручну

Тепер поєднуємо обидві ідеї: приймаємо функцію як аргумент і повертаємо нову функцію, яка спочатку перевіряє роль, а потім, якщо перевірка пройдена, викликає оригінальну функцію. Це і є суть декоратора — Python лише додає для нього зручний синтаксис (наступний розділ).

In [6]:
def require_admin(func):
    """Обгортка: перевіряє роль перед виконанням func."""

    def wrapper():
        if current_user["role"] != "admin":
            print("Доступ заборонено. Потрібна роль: admin")
            return
        func()

    return wrapper  # require_admin повернула — wrapper лишається замиканням над func


def delete_everything():
    print("Видалено все.")


safe_delete = require_admin(delete_everything)  # func у замиканні wrapper — це delete_everything

current_user = {"name": "Гість", "role": "guest"}
safe_delete()

current_user = {"name": "Адмін", "role": "admin"}
safe_delete()

Доступ заборонено. Потрібна роль: admin
Видалено все.


### Типова помилка: `func()` поза `wrapper`

Найчастіша помилка при першій спробі написати обгортку — випадково дедентнути виклик `func()` так, що він опиняється всередині зовнішньої функції, а не всередині `wrapper`. Тоді Python виконує `func()` **одразу, в момент створення обгортки**, а не пізніше, коли хтось викличе результат. Побачити це найлегше, якщо реально виконати зламаний варіант:

In [7]:
# НЕПРАВИЛЬНО: func() на відступі require_admin, а не wrapper
def broken_admin(func):
    def wrapper():
        if current_user["role"] != "admin":
            print("Доступ заборонено.")
        return
    func()          # ← мав бути ВСЕРЕДИНІ wrapper, а не тут
    return wrapper


def demo_action():
    print("Виконано!")


current_user = {"name": "Гість", "role": "guest"}  # явно фіксуємо стан для цієї демонстрації

print("--- створюємо обгортку ---")
protected = broken_admin(demo_action)   # demo_action встигає виконатись ТУТ

print("--- викликаємо protected() як гість ---")
protected()

print("--- викликаємо protected() як адмін ---")
current_user["role"] = "admin"
protected()

--- створюємо обгортку ---
Виконано!
--- викликаємо protected() як гість ---
Доступ заборонено.
--- викликаємо protected() як адмін ---


`"Виконано!"` з'являється ще на кроці `broken_admin(demo_action)` — до будь-якої перевірки ролі, і навіть до того, як `protected` взагалі викликали. А коли `protected()` викликають як адмін, нічого не відбувається: `wrapper` завжди тільки перевіряє роль і повертає `None`, бо `func()` "втік" з-під її відступу. Порівняйте з робочою версією вище (комірка з `require_admin`) — там `func()` лишається на відступі `wrapper`, тому виконується **тільки в момент виклику** `safe_delete()`, і тільки якщо перевірка пройдена.

### 6. Синтаксис `@decorator`

`safe_delete = require_admin(delete_everything)` і

```python
@require_admin
def delete_everything():
    ...
```

виконують **рівно одне й те саме**: визначають `delete_everything`, одразу передають її в `require_admin` і зберігають результат назад під іменем `delete_everything`. `@` — це синтаксичний цукор для цього виклику, не нова механіка.

### 7. Узагальнений декоратор з параметрами

`require_admin` перевіряє тільки одну конкретну роль. Щоб декоратор приймав список ролей як параметр (`@require_role("admin", "moderator")`), потрібен ще один рівень вкладеності — декоратор-фабрика.

| Рівень | Що робить |
|---|---|
| `require_role(*allowed_roles)` | Зовнішня функція-фабрика: приймає список дозволених ролей і повертає `decorator` |
| `decorator(func)` | Приймає функцію, яку потрібно захистити, повертає `wrapper` |
| `wrapper(*args, **kwargs)` | Те, що реально виконується замість оригінальної функції |
| `*args, **kwargs` | Передають усі аргументи виклику далі в `func`, незалежно від того, скільки їх і які вони — інакше декоратор підійшов би лише для функцій без аргументів |
| `functools.wraps(func)` | Копіює `__name__` і докстрінг `func` на `wrapper` — без цього рядка `delete_post.__name__` став би `"wrapper"`, а не `"delete_post"` |

In [8]:
import functools


def require_role(*allowed_roles):
    """Декоратор-фабрика: повертає декоратор, що дозволяє виклик лише переліченим ролям."""

    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_user["role"] not in allowed_roles:
                print(f"Доступ заборонено. Потрібна роль: {' або '.join(allowed_roles)}")
                return
            return func(*args, **kwargs)

        return wrapper

    return decorator


@require_role("admin")
def sample_delete(post_id):
    """Видаляє пост."""
    print(f"Видалено пост #{post_id}")


print(sample_delete.__name__, "—", sample_delete.__doc__)  # functools.wraps зберіг ім'я і докстрінг

current_user = {"name": "Гість", "role": "guest"}
sample_delete(1)

current_user = {"name": "Адмін", "role": "admin"}
sample_delete(1)

sample_delete — Видаляє пост.
Доступ заборонено. Потрібна роль: admin
Видалено пост #1


## 🎮 PREDICT / RUN / INVESTIGATE — рефакторинг доступу через декоратор

### PREDICT

Нижче всі шість функцій блогу переписані через `@require_role(...)`. Перш ніж запускати клітинки тестів далі — дайте відповідь:

1. `publish_post` матиме `@require_role("admin", "moderator")`. Чи виконається `publish_post(42)`, викликана від імені користувача з роллю `moderator`?
2. Та сама `publish_post`, викликана від імені `user` (не `admin` і не `moderator`) — що станеться?

<details>
<summary>Відповідь</summary>

1. Так — `"moderator"` є в списку дозволених ролей, `wrapper` викличе оригінальну функцію.
2. `"user"` немає в списку `("admin", "moderator")` — `wrapper` надрукує повідомлення про відмову і поверне `None`, не викликаючи тіло `publish_post`.

</details>

### RUN

In [9]:
@require_role("guest", "user", "admin", "moderator")
def view_post(post_id):
    print(f"{current_user['name']} переглядає пост #{post_id}")


@require_role("user", "admin")
def create_post(title):
    print(f"{current_user['name']} створює пост: '{title}'")


@require_role("user", "admin", "moderator")
def edit_post(post_id):
    print(f"{current_user['name']} редагує пост #{post_id}")


@require_role("admin", "moderator")
def publish_post(post_id):
    print(f"{current_user['name']} публікує пост #{post_id}")


@require_role("admin")
def delete_post(post_id):
    print(f"{current_user['name']} видаляє пост #{post_id}")


@require_role("admin")
def archive_post(post_id):
    print(f"{current_user['name']} архівує пост #{post_id}")


print("Декоратори застосовано до всіх шести функцій.")

Декоратори застосовано до всіх шести функцій.


In [10]:
current_user = {"name": "Гість Анонім", "role": "guest"}
print(f"=== {current_user['name']} ({current_user['role']}) ===")
view_post(42)
create_post("Мій пост")
delete_post(42)

=== Гість Анонім (guest) ===
Гість Анонім переглядає пост #42
Доступ заборонено. Потрібна роль: user або admin
Доступ заборонено. Потрібна роль: admin


In [11]:
current_user = {"name": "Марко", "role": "moderator"}
print(f"=== {current_user['name']} ({current_user['role']}) ===")
view_post(42)
edit_post(42)
publish_post(42)
delete_post(42)

=== Марко (moderator) ===
Марко переглядає пост #42
Марко редагує пост #42
Марко публікує пост #42
Доступ заборонено. Потрібна роль: admin


In [12]:
current_user = {"name": "Адмін", "role": "admin"}
print(f"=== {current_user['name']} ({current_user['role']}) ===")
view_post(42)
create_post("Важливий анонс")
edit_post(42)
publish_post(42)
delete_post(42)
archive_post(42)

=== Адмін (admin) ===
Адмін переглядає пост #42
Адмін створює пост: 'Важливий анонс'
Адмін редагує пост #42
Адмін публікує пост #42
Адмін видаляє пост #42
Адмін архівує пост #42


### INVESTIGATE

Порівняйте тіло `delete_post` до і після рефакторингу:

**До:**
```python
def delete_post(post_id):
    if current_user["role"] not in ["admin"]:
        print("Доступ заборонено")
        return
    print(f"{current_user['name']} видаляє пост #{post_id}")
```

**Після:**
```python
@require_role("admin")
def delete_post(post_id):
    print(f"{current_user['name']} видаляє пост #{post_id}")
```

Тіло функції тепер містить лише те, що вона справді робить — видаляє пост. Перевірка доступу перенесена в один рядок над визначенням і в одне місце (`require_role`), спільне для всіх шести функцій.

## 🛠️ MODIFY — додай роль `superuser`

Потрібно додати нову роль `superuser`, яка має ті самі права, що й `admin`, для `delete_post` і `archive_post`. Завдяки декоратору це зміна одного рядка в кожній із двох функцій — без редагування їхніх тіл.

Допиши `@require_role(...)` у клітинці нижче так, щоб `superuser` міг видаляти й архівувати пости нарівні з `admin`.

In [13]:
# YOUR CODE HERE — допиши allowed_roles так, щоб superuser мав ті самі права, що admin
# BEGIN SOLUTION
@require_role("admin", "superuser")
def delete_post_v3(post_id):
    print(f"{current_user['name']} видаляє пост #{post_id}")


@require_role("admin", "superuser")
def archive_post_v3(post_id):
    print(f"{current_user['name']} архівує пост #{post_id}")
# END SOLUTION


current_user = {"name": "Супер Юзер", "role": "superuser"}
delete_post_v3(99)
archive_post_v3(99)

current_user = {"name": "Звичайний user", "role": "user"}
delete_post_v3(99)  # має бути заблоковано

Супер Юзер видаляє пост #99
Супер Юзер архівує пост #99
Доступ заборонено. Потрібна роль: admin або superuser


## 🛠️ CREATE

Декоратори корисні не лише для перевірки прав доступу — будь-яка поведінка, яку потрібно додати "до" або "після" виклику функції, не змінюючи саму функцію, є кандидатом на декоратор. Класичні приклади: логування, вимірювання часу виконання, кешування, повторна спроба при помилці.

### Розібраний приклад: `timer`

In [14]:
import time


def timer(func):
    """Вимірює й друкує час виконання func."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        end = time.time()
        print(f"{func.__name__} виконалась за {end - start:.4f} сек")
        return result

    return wrapper


@timer
def heavy_calculation(n):
    """Симулює важкі обчислення — сумує n чисел по одному."""
    return sum(range(n))


@timer
def fast_calculation(n):
    """Та сама сума через формулу Гауса — O(1) замість O(n)."""
    return n * (n - 1) // 2


print(heavy_calculation(1_000_000))
print(fast_calculation(1_000_000))

heavy_calculation виконалась за 0.0073 сек
499999500000
fast_calculation виконалась за 0.0000 сек
499999500000


### Самостійно: `log_call`

Напиши декоратор `log_call`, який перед викликом функції друкує її ім'я та аргументи, а після виклику повертає результат без змін. Формат рядка друку: `Виклик: <ім'я>(<аргументи через кому>)`.

Використай `*args`, `**kwargs` і `functools.wraps` — так само, як у `timer`.

In [15]:
# YOUR CODE HERE
# BEGIN SOLUTION
def log_call(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        all_args = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()]
        print(f"Виклик: {func.__name__}({', '.join(all_args)})")
        return func(*args, **kwargs)

    return wrapper
# END SOLUTION


@log_call
def add(a, b):
    return a + b


result = add(2, 3)
print("Результат:", result)

result2 = add(a=5, b=7)
print("Результат:", result2)

assert result == 5
assert result2 == 12
print("OK")

Виклик: add(2, 3)
Результат: 5
Виклик: add(a=5, b=7)
Результат: 12
OK


## 🔄 TRANSFER — той самий патерн, інший домен

Система бронювання квитків у кінотеатрі, з тими самими трьома ролями за змістом, що й у блозі:

- `guest` — може тільки переглядати розклад сеансів;
- `customer` — може ще й бронювати та скасовувати квитки;
- `admin` — може ще й оформлювати повернення коштів.

Напиши декоратор-фабрику `require_role_cinema(*allowed_roles)` — структурно ідентичну `require_role` вище (та сама схема: фабрика → `decorator(func)` → `wrapper(*args, **kwargs)` → `functools.wraps`), але для цієї системи — і застосуй її до чотирьох функцій нижче.

In [16]:
current_session = {}

# YOUR CODE HERE
# BEGIN SOLUTION
def require_role_cinema(*allowed_roles):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            if current_session["role"] not in allowed_roles:
                print(f"Доступ заборонено. Потрібна роль: {' або '.join(allowed_roles)}")
                return
            return func(*args, **kwargs)
        return wrapper
    return decorator


@require_role_cinema("guest", "customer", "admin")
def view_showtimes():
    print("Розклад сеансів: 14:00, 17:30, 20:00")


@require_role_cinema("customer", "admin")
def book_ticket(showtime):
    print(f"{current_session['name']} забронював(-ла) квиток на сеанс {showtime}")


@require_role_cinema("customer", "admin")
def cancel_booking(showtime):
    print(f"{current_session['name']} скасував(-ла) бронювання на сеанс {showtime}")


@require_role_cinema("admin")
def issue_refund(showtime):
    print(f"{current_session['name']} оформив(-ла) повернення коштів за сеанс {showtime}")
# END SOLUTION


current_session = {"name": "Гість", "role": "guest"}
view_showtimes()
book_ticket("17:30")

current_session = {"name": "Марія", "role": "customer"}
view_showtimes()
book_ticket("17:30")
issue_refund("17:30")

current_session = {"name": "Адмін", "role": "admin"}
issue_refund("17:30")

Розклад сеансів: 14:00, 17:30, 20:00
Доступ заборонено. Потрібна роль: customer або admin
Розклад сеансів: 14:00, 17:30, 20:00
Марія забронював(-ла) квиток на сеанс 17:30
Доступ заборонено. Потрібна роль: admin
Адмін оформив(-ла) повернення коштів за сеанс 17:30


## ✅ Самоперевірка (5 запитань)

> Примітка: оригінальний шаблон уроку (`PY_UKR_L09_filled_template.docx`) — бінарний `.docx`-файл, недоступний для автоматичного читання в цьому середовищі. Питання нижче побудовані на тому самому матеріалі (closures/`nonlocal`, wrapper), а не є дослівним перекладом файлу.

**1.** У функції `make_counter` з розділу «Замикання» видалили рядок `nonlocal count`. Що станеться при виклику `increment()`?

<details><summary>Відповідь</summary><code>UnboundLocalError</code> — рядок <code>count += 1</code> без <code>nonlocal</code> трактується Python як створення нової <b>локальної</b> змінної <code>count</code> усередині <code>increment</code>, а читання значення відбувається до її створення (бо <code>+=</code> спершу читає поточне значення).</details>

**2.** Навіщо `wrapper` у декораторі приймає `*args, **kwargs`, а не просто `(post_id)`?

<details><summary>Відповідь</summary>Щоб один і той самий декоратор підходив для функцій з будь-якою кількістю та типом аргументів (наприклад, <code>view_showtimes()</code> без аргументів і <code>book_ticket(showtime)</code> з одним) — <code>wrapper</code> не повинен знати наперед сигнатуру функції, яку загортає.</details>

**3.** Що станеться з `delete_post.__name__`, якщо прибрати `@functools.wraps(func)` з `decorator`?

<details><summary>Відповідь</summary>Стане <code>"wrapper"</code> замість <code>"delete_post"</code> — без <code>functools.wraps</code> задекорована функція втрачає ім'я та докстрінг оригіналу, підмінюючись іменем внутрішньої функції <code>wrapper</code>.</details>

**4.** Чим декоратор без параметрів (`def require_admin(func): ...`) відрізняється рівнями вкладеності від декоратора-фабрики з параметрами (`def require_role(*roles): ...`)?

<details><summary>Відповідь</summary>Декоратор без параметрів має два рівні: <code>require_admin(func)</code> одразу повертає <code>wrapper</code>. Фабрика має три рівні: <code>require_role(*roles)</code> повертає <code>decorator(func)</code>, яка вже повертає <code>wrapper</code> — зайвий рівень потрібен, щоб спершу "запам'ятати" параметри (<code>roles</code>), а вже потім отримати функцію, яку загортаємо.</details>

**5.** Логіку перевірки ролі можна було б викликати як звичайну функцію на початку кожного `def` (як у наївному варіанті на початку уроку). Коли декоратор — краще рішення, ніж такий виклик?

<details><summary>Відповідь</summary>Коли та сама поведінка ("до" або "після" виклику) потрібна багатьом функціям одразу, і хочеться, щоб вона була описана в одному місці, а не повторювалась у тілі кожної функції — це й називають cross-cutting concern (наскрізна турбота: доступ, логування, таймінг, кешування). Для одноразової перевірки всередині однієї конкретної функції звичайний виклик усередині тіла — цілком достатнє і простіше рішення.</details>

## Шпаргалка

```python
import functools

# Декоратор без параметрів
def my_decorator(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        # робимо щось ДО
        result = func(*args, **kwargs)
        # робимо щось ПІСЛЯ
        return result
    return wrapper


# Декоратор-фабрика — З параметрами
def my_decorator_with_args(param):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # param доступний тут через замикання
            return func(*args, **kwargs)
        return wrapper
    return decorator


# Використання
@my_decorator
def foo():
    pass

@my_decorator_with_args("значення")
def bar():
    pass
```

## Далі

Урок 10 — «Ітератори й генератори» продовжує тему організації рішення, але вже про ліниву обробку даних: `iterable → iterator → iter() → next() → generator → yield`.